### 5.1.1 使用GPT生成文本

In [1]:
import torch                        # 导入torch库
# from chapter04 import GPTModel      # 从第4章导入GPTModel
from gbt import GPTModel

GPT_CONFIG_124M = {                 # GPT配置字典
    "vocab_size": 50257,            # 词汇表大小
    "context_length": 256,          #A 将上下文长度从1024缩短到256词元
    "emb_dim": 768,                 # 嵌入维度
    "n_heads": 12,                  # 注意力头数量
    "n_layers": 12,                 # 层数
    "drop_rate": 0.1,               #B 可能且常见的是将dropout设置为0。
    "qkv_bias": False               # QKV偏置
}

torch.manual_seed(123)              # 设置随机种子
model = GPTModel(GPT_CONFIG_124M)   # 使用配置初始化模型
model.eval()  

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [2]:
def generate_text_simple(model, idx, max_new_tokens, context_size):  # 定义生成简单文本的函数
    for _ in range(max_new_tokens):  # 循环生成新词元
        idx_cond = idx[:, -context_size:]  # 截取上下文
        with torch.no_grad():  # 禁用梯度计算
            logits = model(idx_cond)  # 通过模型获得logits
        
        logits = logits[:, -1, :]  # 只关注最后一个时间步的logits
        probas = torch.softmax(logits, dim=-1)  # 应用softmax函数获取概率分布
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # 选择具有最高概率的词元ID
        idx = torch.cat([idx, idx_next], dim=1)  # 将生成的词元ID附加到当前上下文
    return idx  # 返回生成的词元序列


代码实现文本生成

In [3]:
import tiktoken                    # 导入tiktoken库
# from chapter04 import generate_text_simple # 从第4章导入generate_text_simple函数

def text_to_token_ids(text, tokenizer):   # 定义text_to_token_ids函数
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'}) # 编码文本，允许特殊词元
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # 添加批次维度
    return encoded_tensor             # 返回编码后的张量

def token_ids_to_text(token_ids, tokenizer): # 定义token_ids_to_text函数
    flat = token_ids.squeeze(0)       # 移除批次维度
    return tokenizer.decode(flat.tolist()) # 解码为文本

start_context = "Every effort moves you" # 设置初始上下文
tokenizer = tiktoken.get_encoding("gpt2") # 获取GPT-2的分词器编码

token_ids = generate_text_simple(   # 调用generate_text_simple函数生成词元ID
    model=model,                    # 模型
    idx=text_to_token_ids(start_context, tokenizer), # 将初始上下文转换为词元ID
    max_new_tokens=10,              # 最大新词元数
    context_size=GPT_CONFIG_124M["context_length"]  # 上下文长度
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer)) # 打印生成的文本


Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


根据输出结果，很明显模型尚未生成连贯的文本，因为它还没有经过训练。要定义使文本 “连贯”或“高质量” 的标准，我们必须实现一种数值方法来评估生成的内容。这种方法将使我们能够在整个训练过程中监控和提高模型的性能。

In [4]:
file_path = "the-verdict.txt"  # 文件路径
with open(file_path, "r", encoding="utf-8") as file:  # 以读模式打开文件
    text_data = file.read()  # 读取文件内容


In [ ]:
# 加载数据集后，我们可以检查数据集中的字符和词元数量：
total_characters = len(text_data)  # 计算总字符数
total_tokens = len(tokenizer.encode(text_data))  # 计算总词元数
print("Characters:", total_characters)  # 打印字符数
print("Tokens:", total_tokens)  # 打印词元数

Characters: 20479
Tokens: 5145


In [6]:
train_ratio = 0.90  # 训练集比例
split_idx = int(train_ratio * len(text_data))  # 计算分割索引
train_data = text_data[:split_idx]  # 获取训练数据
val_data = text_data[split_idx:]  # 获取验证数据


In [8]:
from create_dataloader_v1 import  create_dataloader_v1

torch.manual_seed(123)  # 设置随机种子

train_loader = create_dataloader_v1(
    train_data,  # 训练数据
    batch_size=2,  # 批大小
    max_length=GPT_CONFIG_124M["context_length"],  # 最大长度
    stride=GPT_CONFIG_124M["context_length"],  # 步幅
    drop_last=True,  # 丢弃最后一个不完整批次
    shuffle=True,  # 是否打乱数据
    num_workers=0  # 工作线程数
)

val_loader = create_dataloader_v1(
    val_data,  # 验证数据
    batch_size=2,  # 批大小
    max_length=GPT_CONFIG_124M["context_length"],  # 最大长度
    stride=GPT_CONFIG_124M["context_length"],  # 步幅
    drop_last=False,  # 不丢弃最后一个不完整批次
    shuffle=False,  # 是否打乱数据
    num_workers=0  # 工作线程数
)

tiktoken version: 0.7.0


In [9]:
print("Train loader:")  # 打印训练加载器
for x, y in train_loader:
    print(x.shape, y.shape)  # 打印每个批次的形状

print("\nValidation loader:")  # 打印验证加载器
for x, y in val_loader:
    print(x.shape, y.shape)  # 打印每个批次的形状


Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [10]:
def calc_loss_batch(input_batch, target_batch, model, device):  # 定义计算批次损失的函数
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)  # 将输入和目标批次转移到设备上
    logits = model(input_batch)  # 模型计算logits
    loss = torch.nn.functional.cross_entropy(  # 计算交叉熵损失
        logits.flatten(0, 1), target_batch.flatten()  # 展平logits和目标批次
    )
    return loss  # 返回损失
